In [1]:
import pandas as pd
from glob import glob
from pathlib import Path

import mmdet
import mmcv
import os

In [2]:
from mmcv import Config
cfg = Config.fromfile('/ceph/hpc/home/euerikl/projects/fastighet/models/configs/test_cfg_dump.py')
#cfg.dump('/ceph/hpc/home/euerikl/projects/fastighet/models/configs/test_cfg_dump.py')

In [3]:
from mmdet.apis import inference_detector, init_detector
checkpoint = '/ceph/hpc/home/euerikl/projects/fastighet/models/cascade_rcnn_1/latest.pth'
model = init_detector(cfg, checkpoint, device='cuda:0')

load checkpoint from local path: /ceph/hpc/home/euerikl/projects/fastighet/models/cascade_rcnn_1/latest.pth


In [4]:
#skriv om till ett skript, pröva med flera gpu:er, och lägg batcherna på olika gpu:er 

imgs = glob('/ceph/hpc/scratch/user/euerikl/data/fastighet/miljonsetet/all_batches/10018814/*.tif')
imgs_chunks = [imgs[x:x+8] for x in range(0, len(imgs), 8)]

output_path = os.path.join('/ceph/hpc/home/euerikl/projects/fastighet/output', '10018814')
Path(output_path).mkdir(exist_ok=True)

batch_json = dict()

for chunk in imgs_chunks:
    results = inference_detector(model, chunk)
    for i, img_p in enumerate(chunk):
        
        img_name = Path(img_p).stem
        batch_json[img_name] = []

        im = mmcv.imread(img_p)

        for j,bb in enumerate(results[i][0]):

            batch_json[img_name].append(dict())
            batch_json[img_name][j]['det_prob'] = str(bb[4])

            
            file_name = Path(img_p).stem + '_' + str(j).zfill(3) + '.tif'
            file_out = os.path.join(output_path, file_name)

            cropped_img = mmcv.imcrop(im, bb[0:4])
            mmcv.imwrite(cropped_img, file_out)

In [6]:
import json

with open('/ceph/hpc/home/euerikl/projects/fastighet/output/batch_10018814.json', 'w', encoding='utf8') as f:
    s = json.dumps(batch_json, indent = 4, ensure_ascii=False, sort_keys=True)
    f.write(str(s))

In [21]:
print(results[3][0][0])

[3.3322190e+03 3.3792383e+02 4.4316992e+03 4.8483893e+02 9.8655635e-01]


In [18]:
Path(imgs[0]).stem

'10018814_00000292'